<a href="https://colab.research.google.com/github/otis222/genai-week7-rag/blob/main/RAG01_%E9%81%B8%E8%AA%B2%E8%BB%8D%E5%B8%AB_%E5%BB%BA%E7%AB%8B%E5%90%91%E9%87%8F%E8%B3%87%E6%96%99%E5%BA%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG 01｜大同大學選課軍師：建立向量資料庫

> 第七週作業：打造自己的 RAG 對話機器人（程式 1／2）
> 參考老師的 [RAG_01_打造你的_RAG_系統](https://github.com/yenlung/Generative-AI)，換成我自己的主題與資料。

**這份程式做什麼？** 讀入我整理好的原始資料 `rag-data.zip`（一篇一個 TXT），每篇轉成一個向量，用 FAISS 建立向量資料庫，最後打包成 **`vector_embedding.zip`** 給第二份程式（RAG02）使用。

```
rag-data.zip（原始 TXT）→ 擷取正文與 metadata → EmbeddingGemma 2 向量化 → FAISS 索引 → vector_embedding.zip
```

📌 **檔案命名說明**：依照老師投影片（p.51）的流程，原始資料叫 `rag-data.zip`，向量資料庫叫 `vector_embedding.zip`。作業說明中「RAG01 儲存、RAG02 讀入並解壓縮」的 ZIP，指的就是 `vector_embedding.zip`。

**操作方式：由上往下一格一格執行。** 本程式不呼叫 LLM，也不需要 API 金鑰。可在「執行階段 → 變更執行階段類型」選 GPU；沒有 GPU 用 CPU 也能跑，只是比較慢。

## 我的主題與資料

**主題：大同大學選課軍師**：把大同大學的選課相關規章做成 RAG 知識庫，讓「選課軍師」回答加簽、超修、期中退選、暑修、通識學分等問題。每間學校的選課規定都不一樣，一般 LLM 不可能記得大同大學的細節，正好適合用 RAG。

**資料來源（皆為大同大學官方公開資料）**

| 來源 | 內容 |
|---|---|
| [大同大學學生選課辦法](https://tturule.ttu.edu.tw/rulelist/showcontent.php?rid=398)（113.5.3 教務會議修正，**主要來源**） | 選課流程、學分上下限、超修、加簽、擋修、衝堂、跨部與校際選課 |
| [大同大學期中退選課程辦法](https://tturule.ttu.edu.tw/rulelist/showcontent.php?rid=329) | 申請方式、每學期兩科上限、成績單記 W |
| [大同大學學生暑期補修課程規定](https://tturule.ttu.edu.tw/rulelist/showcontent.php?rid=607) | 暑修資格、每期 7 學分上限、繳費與退費 |
| [校訂共同必修基礎課程暨博雅通識課程修課規定](https://tturule.ttu.edu.tw/rulelist/showcontent.php?rid=1116) | 115 學年度起入學的共同必修與博雅通識 |
| [註冊課務組選課 Q&A](https://curri.ttu.edu.tw/p/412-1033-482.php) | 加簽、超修、校際選課、選課不合規定的處理 |
| [115 學年度行事曆](https://reg.ttu.edu.tw/p/405-1032-42221,c910.php)、[新生選課專區](https://b0225.ttu.edu.tw/p/406-1078-36651,r11.php?Lang=zh-tw) | 選課、加退選、期中退選日期與網路選課步驟 |

**資料怎麼做出來的？** 依老師上課的建議「一個意思＝一篇文件」：① 找到官方原文 → ② 請生成式 AI 依「一個條文或一個 Q&A 一篇」切成 27 個 UTF-8 TXT，白話改寫但保留數字、日期與條號 → ③ 對照原文逐篇檢查 → ④ 打包成 `rag-data.zip`。

**一篇資料長這樣**（上面是 metadata，只有 `【內容】` 會和分類、標題一起做 embedding）：

```
標題：期中退選的限制：每學期最多兩科、微型課程不能退
分類：期中退選
資料類型：學校規章的教學改寫摘要（非法規全文、非官方解釋）
發布單位：大同大學教務處
來源網址：https://tturule.ttu.edu.tw/rulelist/showcontent.php?rid=329
法規依據：大同大學期中退選課程辦法 第二條、第五條、第六條（106年3月30日教務會議修正通過）；大同大學學生選課辦法 第5條第1項
資料整理日期：2026-10-21

【內容】
微型課程不能辦理期中退選。期中退選每學期以兩科為限，退選的學分不計入該學期所修學分數。學生申請期中退選後，當學期修習的學分數仍然要符合最低應修學分數的規定（依選課辦法第5條，學士班一、二、三年級每學期最低13學分，四年級最低9學分）；……

【檢索練習問題】
我這學期修14學分，可以期中退選一門3學分的課嗎？

【提醒】
這是為 RAG 作業改寫的白話摘要；完整條文與最新修正，請以大同大學「法規與會議紀錄管理系統」公告為準。
```

✏️ 跟老師範例相比，我多了 `分類` 和 `法規依據` 兩個欄位：分類會放進 embedding 文字，法規依據放在 metadata，讓軍師回答時可以說出「依選課辦法第 9 條」。

✏️ **檢查檢索結果後的修改**：測試時發現「修 14 學分能不能期中退選 3 學分」這題只找得到期中退選的規定，找不到「最低 13 學分」那篇，所以在期中退選那篇補上最低學分數，讓一篇文件自己就看得懂（老師投影片 p.22：需要加入上下文嗎？）。另外把內容太雜的「特殊身分」拆成兩篇，並在校際選課那篇加上「其他大學」的說法，讓「可以去台大修課嗎？」這種問法也找得到。

## 1. 安裝套件

與老師範例相同，固定套件版本，方便重現。安裝後如果出現「重新啟動工作階段」的提示，請重新啟動，再從下一格繼續。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 老師的固定套件

📌 每份作業都要引入的標準套件。

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from urllib.parse import urlparse, parse_qs
import importlib.metadata as package_metadata
import hashlib
import io
import json
import re
import tempfile
import zipfile

import requests
import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

## 3. 集中設定

✏️ **我的修改：新增 `RAG_DATA_URL`。** 老師的範例要手動上傳 ZIP；我把 `rag-data.zip` 放到雲端，讓程式直接下載，**助教不用另外準備檔案就能從頭執行**。如果把網址改成空字串 `""`，就會改回手動上傳。

其他設定沿用老師範例：EmbeddingGemma 2、文件與問題使用不同的前綴（prefix）。

In [ ]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4
EXPECTED_DOCUMENT_COUNT = None  # 不強制篇數
EXPORT_ZIP_NAME = "vector_embedding.zip"

# ✏️ 原始資料 rag-data.zip 的下載網址（Google 雲端硬碟分享連結或 GitHub 連結都可以）
RAG_DATA_URL = "https://raw.githubusercontent.com/otis222/genai-week7-rag/main/rag-data.zip"

# text 本身已含分類與標題；這裡不再從 metadata 添加標題。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

## 4. 讀入原始資料 rag-data.zip

✏️ **我的修改：從網址下載。** `to_direct_download_url()` 會把 Google 雲端硬碟的「分享連結」自動轉成「可直接下載的網址」（作業要求的「雲端轉成可下載的網址」），`download_zip()` 會檢查下載到的是不是 ZIP；如果雲端沒有開共用權限，會拿到登入網頁而不是 ZIP，這時會跳出清楚的錯誤提示。

解壓縮的部分沿用老師的安全寫法：只解開 TXT，而且確保 ZIP 裡的路徑不會寫到暫存資料夾外面。

In [ ]:
def to_direct_download_url(url):
    """把 Google 雲端硬碟或 GitHub 的分享連結轉成可直接下載的網址；其他網址原樣傳回。"""
    parsed = urlparse(url)
    if parsed.netloc.endswith("drive.google.com"):
        match = re.search(r"/d/([A-Za-z0-9_-]+)", parsed.path)
        file_id = match.group(1) if match else parse_qs(parsed.query).get("id", [None])[0]
        if not file_id:
            raise ValueError("看不出 Google 雲端硬碟的檔案 ID，請確認分享連結。")
        return f"https://drive.google.com/uc?export=download&id={file_id}"
    if parsed.netloc == "github.com" and "/blob/" in parsed.path:
        return url.replace("https://github.com/", "https://raw.githubusercontent.com/", 1).replace("/blob/", "/", 1)
    return url


def download_zip(url):
    """下載 ZIP，並確認拿到的真的是 ZIP（雲端權限沒開時，常會拿到登入網頁）。"""
    direct_url = to_direct_download_url(url.strip())
    try:
        response = requests.get(direct_url, timeout=120)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(f"下載失敗，請確認網址與網路：{direct_url}") from exc
    data = response.content
    if not zipfile.is_zipfile(io.BytesIO(data)):
        raise ValueError("下載到的不是 ZIP。請確認雲端硬碟分享設定為「知道連結的任何人」，並貼上正確網址。")
    return data

In [ ]:
if RAG_DATA_URL.strip():
    archive_name = "rag-data.zip"
    archive_bytes = download_zip(RAG_DATA_URL)
    print("已從網址下載：", archive_name, f"({len(archive_bytes):,} bytes)")
else:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("請一次上傳一個資料 ZIP 檔。")
    archive_name = zip_names[0]
    archive_bytes = uploaded[archive_name]

SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
DATA_DIR = Path(tempfile.mkdtemp(prefix="rag_documents_"))

# 僅解開 TXT，並確保 ZIP 中的路徑不會寫到暫存資料夾外。
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = [
        member for member in archive.infolist()
        if not member.is_dir()
        and member.filename.lower().endswith(".txt")
        and not any(part.startswith((".", "__MACOSX")) for part in Path(member.filename).parts)
    ]
    if not members:
        raise ValueError("ZIP 中找不到 TXT；請確認 rag-data.zip 的內容。")
    destinations = set()
    for member in members:
        target = (DATA_DIR / member.filename).resolve()
        if not target.is_relative_to(DATA_DIR.resolve()):
            raise ValueError(f"ZIP 包含不適當的檔案路徑：{member.filename}")
        if target in destinations:
            raise ValueError(f"ZIP 有重複檔名：{member.filename}")
        destinations.add(target)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))

txt_paths = sorted(
    [p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".txt"],
    key=lambda p: p.relative_to(DATA_DIR).as_posix(),
)
print("資料：", archive_name)
print("TXT 數量：", len(txt_paths))
for path in txt_paths:
    print("•", path.relative_to(DATA_DIR).as_posix())
if EXPECTED_DOCUMENT_COUNT is not None and len(txt_paths) != EXPECTED_DOCUMENT_COUNT:
    print(f"提醒：本次為 {len(txt_paths)} 篇，與預期的 {EXPECTED_DOCUMENT_COUNT} 篇不同；仍可繼續。")

## 5. 先看一篇原始文件

我的每篇文件有這些欄位：`標題`、`分類`、`資料類型`、`發布單位`、`來源網址`、`法規依據`、`資料整理日期`、`【內容】`、`【檢索練習問題】`、`【提醒】`。

📌 **不是每個欄位都適合送進 embedding。** 跟老師範例一樣，練習問題不放進向量，不然搜尋會直接命中題目，看不出正文真正的檢索效果。

In [ ]:
SAMPLE_INDEX = 0
sample_path = txt_paths[SAMPLE_INDEX]
try:
    sample_raw = sample_path.read_text(encoding="utf-8-sig")
except UnicodeDecodeError as exc:
    raise ValueError(f"{sample_path.name} 不是可讀取的 UTF-8 文字。") from exc
print("檔名：", sample_path.name)
print("=" * 50)
print(sample_raw)

## 6. 可修改函式一：取出要做 embedding 的文字

✏️ **我的修改：把「分類」也放進向量文字。** 老師的版本取「標題＋【內容】」；我的資料有分類欄位，所以組成 `【分類】標題 + 正文`。這樣問比較籠統的問題（例如只問到某個類別）時，也比較容易找到同一類的文件。

這個回傳值會原樣保存，之後檢索到時也是把這段文字交給 LLM，確保「搜尋用的文字」和「回答用的文字」是同一份。

In [ ]:
def extract_rag_text(raw_text):
    """取出【分類】標題與正文；練習問題和提醒不會混入。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    title_match = re.search(r"^標題[：:]\s*(.+)$", normalized, flags=re.MULTILINE)
    category_match = re.search(r"^分類[：:]\s*(.+)$", normalized, flags=re.MULTILINE)
    content_match = re.search(
        r"^【內容】[ \t]*\n(?P<body>.*?)(?=^【[^】\n]+】|\Z)",
        normalized,
        flags=re.MULTILINE | re.DOTALL,
    )
    if content_match is None or not content_match.group("body").strip():
        raise ValueError("找不到非空的【內容】正文；請修改 extract_rag_text() 以配合文件格式。")
    title = title_match.group(1).strip() if title_match else ""
    category = category_match.group(1).strip() if category_match else ""
    heading = f"【{category}】{title}" if category else title
    body = content_match.group("body").strip()
    return f"{heading}\n\n{body}" if heading else body

sample_rag_text = extract_rag_text(sample_raw)
print(sample_rag_text)
print("\n擷取後字元數：", len(sample_rag_text))

## 7. 可修改函式二：取出 metadata

✏️ **我的修改：增加 `category`（分類）、`publisher`（發布單位）與 `basis`（法規依據）欄位。** metadata 不會送進 embedding；RAG02 會用它顯示「資料來源」讓使用者點回原始網頁查證，而 `basis` 會連同文件交給 LLM，讓軍師回答時能說出「依選課辦法第 9 條」。找不到的欄位用 `None`，不自己捏造。

In [ ]:
def extract_metadata(raw_text, filename):
    """取出描述資訊；換資料時修改這裡。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    def read_field(name):
        match = re.search(rf"^{re.escape(name)}[：:][ \t]*(.*)$", normalized, flags=re.MULTILINE)
        return match.group(1).strip() or None if match else None

    return {
        "title": read_field("標題") or Path(filename).stem,
        "category": read_field("分類"),
        "publisher": read_field("發布單位"),
        "basis": read_field("法規依據"),
        "source_url": read_field("來源網址"),
        "date": read_field("資料整理日期"),
        "filename": filename,
        "document_type": read_field("資料類型") or "未標示",
    }

sample_metadata = extract_metadata(sample_raw, sample_path.relative_to(DATA_DIR).as_posix())
print(json.dumps(sample_metadata, ensure_ascii=False, indent=2))

## 8. 批次整理成統一格式

每篇得到 `doc_id`、`text`、`metadata`。檔案先排序再編號，所以只要資料不變，編號就不變。

In [ ]:
documents = []
for number, path in enumerate(txt_paths, start=1):
    filename = path.relative_to(DATA_DIR).as_posix()
    try:
        raw_text = path.read_text(encoding="utf-8-sig")
        if "\ufffd" in raw_text:
            raise ValueError("文字含有替代字元，請檢查原始編碼。")
        rag_text = extract_rag_text(raw_text)
        metadata = extract_metadata(raw_text, filename)
    except (ValueError, UnicodeDecodeError) as exc:
        raise ValueError(f"處理失敗：{filename}；原因：{exc}") from exc
    documents.append({"doc_id": f"doc_{number:03d}", "text": rag_text, "metadata": metadata})

if not documents:
    raise ValueError("沒有可建立索引的文件。")
index_doc_ids = [doc["doc_id"] for doc in documents]
if len(set(index_doc_ids)) != len(index_doc_ids):
    raise ValueError("doc_id 重複，請檢查編號規則。")

summary = pd.DataFrame([
    {"doc_id": doc["doc_id"], "分類": doc["metadata"]["category"], "標題": doc["metadata"]["title"],
     "字元數": len(doc["text"]), "來源": doc["metadata"]["source_url"]}
    for doc in documents
])
display(summary)
for doc in documents:
    missing = [key for key in ("source_url", "date") if not doc["metadata"].get(key)]
    if missing:
        print("提醒：", doc["doc_id"], "缺少", ", ".join(missing))
print("各分類篇數：")
display(summary["分類"].value_counts().rename_axis("分類").reset_index(name="篇數"))
print("第一筆統一格式：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))

## 9. 載入 EmbeddingGemma 2

先查詢模型目前的確切版本（commit）再載入，並把版本記錄到設定檔，讓 RAG02 用**完全相同**的模型把問題轉成向量。只啟用文字部分。首次下載需要等待。

In [ ]:
def embedding_dimension(model):
    # 新版 sentence-transformers 把 get_sentence_embedding_dimension 改名為 get_embedding_dimension
    getter = getattr(model, "get_embedding_dimension", None) or model.get_sentence_embedding_dimension
    return int(getter())

try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", embedding_dimension(model))

## 10. 檢查輸入長度，避免被默默截斷

📌 文件用 `title: none | text: {text}`，問題用 `task: search result | query: {question}`，這是 EmbeddingGemma 區分「內容」與「問題」的前綴。我的文件比老師範例長一些，所以這一步很重要：超過上限就停下來，不會只取前半段。

In [ ]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))

if TOKEN_LIMIT <= 0:
    raise ValueError("輸入長度上限必須大於0。")

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
length_table = pd.DataFrame({"doc_id": index_doc_ids, "tokens_含前綴": token_counts})
display(length_table.describe().T)
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下文件超過 {TOKEN_LIMIT} tokens，請先調整資料：{long_docs}")
print("使用的 token 上限：", TOKEN_LIMIT, "／最長的一篇：", max(token_counts), "tokens")
print("全部文件均未超過上限；不需要截斷。")
print("第一筆真正送入 embedding 的文字：\n", document_inputs[0])

## 11. 產生文件向量

一篇文件對應矩陣中的一列。向量正規化後長度為 1，所以兩個向量的內積就等於 cosine similarity。

In [ ]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != embedding_dimension(model):
    raise ValueError("實際向量維度與模型宣告不符。")
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一篇向量的前10個數字：", embeddings[0, :10])
print("前5篇向量長度：", np.linalg.norm(embeddings[:5], axis=1))

## 12. ✏️ 我加的觀察：文件之間有多像？

用老師固定套件裡的 matplotlib 畫出「文件 × 文件」的 cosine similarity 熱圖。顏色越亮代表兩篇文件的向量越接近。

📌 **觀察重點**：同一個分類的文件通常會比較接近；如果兩篇內容不同的文件相似度卻很高，提問時就可能互相搶位置，這也是 RAG02 一次取回多篇（top-k）的原因。

In [ ]:
similarity = embeddings @ embeddings.T
short_ids = [doc_id.replace("doc_", "") for doc_id in index_doc_ids]

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(similarity, cmap="viridis", vmin=float(similarity.min()), vmax=1.0)
ax.set_xticks(range(len(short_ids)), labels=short_ids, rotation=90, fontsize=7)
ax.set_yticks(range(len(short_ids)), labels=short_ids, fontsize=7)
ax.set_xlabel("doc_id")
ax.set_ylabel("doc_id")
ax.set_title("Cosine similarity between documents")
fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

pairs = [
    (float(similarity[i, j]), documents[i]["doc_id"], documents[i]["metadata"]["title"],
     documents[j]["doc_id"], documents[j]["metadata"]["title"])
    for i in range(len(documents)) for j in range(i + 1, len(documents))
]
pairs.sort(reverse=True)
print("最相似的 5 組文件：")
display(pd.DataFrame(pairs[:5], columns=["相似度", "文件A", "標題A", "文件B", "標題B"]))

## 13. 建立 FAISS 精確搜尋索引

`IndexFlatIP` 會把問題向量和**所有**文件向量算內積。FAISS 回傳的是向量編號，**不是原文**，所以要用 `index_doc_ids` 對回文件。

In [ ]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)

## 14. 定義問題向量化與搜尋函式

問題用同一個 embedding 模型，但加上查詢專用的前綴。這裡只做檢索，還沒有呼叫 LLM。

In [ ]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

## 15. 試查：讀讀看真正取回的內容

✏️ 換成我自己主題的問題。第二個問題是**資料裡沒有答案**的問題：FAISS 還是會找出「最像」的文件。📌 **相似度不是正確率**，所以 RAG02 的 prompt 要求模型在資料不足時老實說。

In [ ]:
TOP_K = 3
for QUESTION in ["我想上的課已經額滿了，要怎麼加簽？", "哪位老師的通識課給分最甜？"]:
    results = search_documents(QUESTION, TOP_K)
    print("\n問題：", QUESTION)
    display(pd.DataFrame([
        {"排名": r["rank"], "doc_id": r["doc_id"], "標題": r["metadata"]["title"],
         "cosine_similarity": round(r["score"], 4)}
        for r in results
    ]))
QUESTION = "我想上的課已經額滿了，要怎麼加簽？"
results = search_documents(QUESTION, TOP_K)
print(f"\n第一名文件全文 [{results[0]['doc_id']}]：")
print(results[0]["text"])
print("來源：", results[0]["metadata"]["source_url"] or "未提供")

## 16. 保存索引、文件、編號對照與設定

RAG02 需要這四個檔案（不能只拿 FAISS 索引）：

| 檔案 | 用途 |
|---|---|
| `index.faiss` | 文件向量與搜尋索引 |
| `documents.json` | 取回後交給 LLM 的正文與來源資訊 |
| `index_doc_ids.json` | FAISS 編號對應的文件編號 |
| `config.json` | 模型版本、前綴、維度、套件版本與檢查碼 |

不保存模型權重或任何金鑰。

In [ ]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "topic": "大同大學選課軍師",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "one_document_one_vector",
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")

## 17. 重新載入並驗證

從剛存好的檔案重新載入，檢查檔案完整性、編號對照，並用**同一個問題向量**比對保存前後的搜尋結果是否一致。

In [ ]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

## 18. 打包並下載 vector_embedding.zip

ZIP 根目錄包含上面四個檔案。執行後瀏覽器會下載 `vector_embedding.zip`。

In [ ]:
from google.colab import files

EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("下載檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")
files.download(str(EXPORT_ZIP))

## 19. 下一步：把 vector_embedding.zip 放上網路

RAG02 會從網址自動下載向量資料庫，助教不用上傳任何檔案。我把 `vector_embedding.zip` 放在和這份筆記本同一個 GitHub repo，RAG02 第 3 節的 `VECTOR_DB_URL` 已經填好它的 raw 網址。

📌 如果修改了資料、重新執行本筆記本，記得把新下載的 `vector_embedding.zip` 上傳覆蓋 repo 裡的舊檔，否則 RAG02 讀到的還是舊版。

也可以改放 Google 雲端硬碟：共用設定改成 **「知道連結的任何人」**，再把分享連結貼到 `VECTOR_DB_URL`，程式會自動轉成可直接下載的網址。權限沒開的話，助教執行時會下載到登入網頁而不是 ZIP（作業註二：權限未開視為無法開啟）。